# Análise de Correlação: Corrupção/Mau Uso de Recursos vs IDH por Cluster

**Projeto:** Compliance Público Baseado em Dados (TCC MBA)  
**Título do TCC:** Correlação entre Repasses Federais e Indicadores Socioeconômicos Municipais

**Autor:** Enok  
**Data:** 2026-04-19

---

## Objetivo

Investigar a correlação entre **indicadores de mau uso de recursos públicos** (sanções por milhão de reais transferidos) e **indicadores de desenvolvimento humano** (renda, alfabetização) **dentro de cada cluster de municípios similares**.

A estratégia de análise por cluster permite **comparar laranja com laranja**: municípios com características socioeconômicas semelhantes são analisados em conjunto, isolando o efeito da eficiência/corrupção do efeito do nível de desenvolvimento.

---

## Hipóteses

1. **H1:** Municípios com mais sanções por real transferido (ineficiência/corrupção) tendem a ter menor IDH dentro do mesmo cluster
2. **H2:** A correlação é mais forte quando controlamos por similaridade socioeconômica (clusters) vs. análise agregada
3. **H3:** Existem clusters com padrões distintos: alguns mostram correlação forte, outros mostram heterogeneidade interna

---

## Métricas de Entrada

**Indicador de Mau Uso/Corrupção (X):**
- `sanctions_per_million_brl_transfers`: Número de sanções por milhão de reais em transferências federais
- Interpretação: quanto maior, mais sanções relativas ao volume de recursos recebidos

**Indicadores de IDH/Desenvolvimento (Y):**
- `avg_income_real_2022_2022_brl`: Renda média real (IPCA 2022) - proxy IDH Renda
- `literacy_rate_2022`: Taxa de alfabetização - proxy IDH Educação
- `income_change_real_pct`: Variação real da renda 2010-2022 - proxy dinâmica de desenvolvimento
- `literacy_change_pp`: Variação da alfabetização (pontos percentuais) 2010-2022

**Controle de Similaridade:**
- `cluster`: Atribuição K-means com 4 clusters baseado em 12 features socioeconômicas

---

## Saídas Esperadas

1. **Tabela de correlação por cluster:** Pearson r e p-value para cada cluster
2. **Índice de vulnerabilidade municipal:** Score combinado para o mapa
3. **Amostra representativa:** Top N cidades por cluster para apresentação
4. **GeoJSON para QGIS:** Dados prontos para mapa coroplético
5. **Dashboard consolidado:** Visualização interativa dos resultados

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## 1. Configuração e Carregamento de Dados

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Adicionar raiz do projeto ao path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Bibliotecas principais
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Estatísticas
from scipy import stats
from scipy.stats import pearsonr, spearmanr

# AWS
import boto3
import tempfile

# Configurar estilo
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Bibliotecas carregadas com sucesso!")

In [ ]:
# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

print(f"Semente fixada em {SEED}")

In [ ]:
# Configuração AWS
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))

print(f"Bucket: {S3_BUCKET_NAME}")
print(f"Profile: {AWS_PROFILE}")

In [ ]:
# Carregar dados da camada Gold local e computar clustering inline
from src.analysis.local_data_loader import LocalGoldDataLoader
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

loader = LocalGoldDataLoader()

# Carregar datasets necessarios
df_state = loader.load_dataset('analysis_compliance')
df_city = loader.load_dataset('analysis_compliance_municipality')
df_cluster = loader.load_dataset('consolidated_clustering')

print(f"Carregado {len(df_state)} linhas de estado")
print(f"Carregado {len(df_city)} linhas de municipios")
print(f"Carregado {len(df_cluster)} linhas de cluster")

# Computar PCA + KMeans clustering (necessario para analise downstream)
feature_cols = [c for c in df_cluster.columns if c.endswith('_norm') or c.startswith('log_')]
X = df_cluster[feature_cols].fillna(0).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3, random_state=42)
pcs = pca.fit_transform(X_scaled)
df_cluster['PC1'] = pcs[:, 0]
df_cluster['PC2'] = pcs[:, 1]
df_cluster['PC3'] = pcs[:, 2]

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(X_scaled)

print(f"Variancia explicada PCA: {pca.explained_variance_ratio_.sum():.1%}")
print(f"Clusters atribuidos: {df_cluster['cluster'].nunique()} clusters unicos")


In [ ]:
# Verificar colunas disponíveis
print("\nColunas disponíveis em df_city:")
for col in sorted(df_city.columns):
    print(f"  - {col}")

## 2. Preparação dos Dados de Análise

In [ ]:
# Merge dos dados de compliance com clusters (incluir todas colunas necessarias de df_cluster)
merge_cols = ['municipality_code', 'cluster', 'PC1', 'PC2', 'PC3', 
              'income_change_real_pct', 'literacy_change_pp',
              'population_change_pct', 'households_change_pct']
# Manter apenas colunas que existem
merge_cols = [c for c in merge_cols if c in df_cluster.columns]

df = df_city.merge(
    df_cluster[merge_cols], 
    on='municipality_code', 
    how='left'
)

print(f"Merge concluido: {len(df):,} municipios com cluster atribuido")

# Verificar municipios sem cluster
no_cluster = df['cluster'].isna().sum()
print(f"Municipios sem cluster: {no_cluster}")


In [ ]:
# Definir variáveis de análise
CORRUPTION_VAR = 'sanctions_per_million_brl_transfers'
IDH_VARS = [
    'avg_income_real_2022_2022_brl',      # Renda (IDH proxy)
    'literacy_rate_2022',                  # Alfabetização
    'income_change_real_pct',              # Dinâmica renda
    'literacy_change_pp',                  # Dinâmica educação
]

# Criar flag para municípios com dados válidos de sanções
df['has_sanctions_data'] = (
    df[CORRUPTION_VAR].notna() & 
    (df[CORRUPTION_VAR] >= 0) &
    df['total_transfers'].notna() &
    (df['total_transfers'] > 0)
)

print("Disponibilidade de dados de sanções:")
print(f"  Com dados válidos: {df['has_sanctions_data'].sum():,}")
print(f"  Sem dados: {(~df['has_sanctions_data']).sum():,}")

# Estatísticas da variável de corrupção
print("\nEstatísticas de sanções por milhão BRL:")
print(df[CORRUPTION_VAR].describe())

## 3. Análise de Correlação por Cluster

In [ ]:
# Função para calcular correlação por cluster
def calcular_correlacao_por_cluster(df, cluster_id, corr_var, idh_var):
    """Calcular correlação entre corr_var e idh_var para um cluster específico."""
    
    subset = df[
        (df['cluster'] == cluster_id) & 
        df['has_sanctions_data'] &
        df[corr_var].notna() & 
        df[idh_var].notna()
    ].copy()
    
    n = len(subset)
    
    if n < 10:
        return {
            'cluster': cluster_id,
            'n_municipios': n,
            'pearson_r': None,
            'pearson_p': None,
            'spearman_r': None,
            'spearman_p': None,
            'interpretacao': 'Amostra insuficiente'
        }
    
    # Remover outliers extremos (>3 desvios padrão)
    z_scores = np.abs(stats.zscore(subset[corr_var]))
    subset_clean = subset[z_scores < 3]
    
    n_clean = len(subset_clean)
    
    if n_clean < 10:
        return {
            'cluster': cluster_id,
            'n_municipios': n_clean,
            'pearson_r': None,
            'pearson_p': None,
            'spearman_r': None,
            'spearman_p': None,
            'interpretacao': 'Dados limpos insuficientes'
        }
    
    # Calcular correlações
    pearson_r, pearson_p = pearsonr(subset_clean[corr_var], subset_clean[idh_var])
    spearman_r, spearman_p = spearmanr(subset_clean[corr_var], subset_clean[idh_var])
    
    # Interpretação
    if pearson_p < 0.001:
        significancia = '***'
    elif pearson_p < 0.01:
        significancia = '**'
    elif pearson_p < 0.05:
        significancia = '*'
    else:
        significancia = 'ns'
    
    return {
        'cluster': cluster_id,
        'n_municipios': n_clean,
        'pearson_r': round(pearson_r, 4),
        'pearson_p': round(pearson_p, 6),
        'spearman_r': round(spearman_r, 4),
        'spearman_p': round(spearman_p, 6),
        'significancia': significancia,
        'interpretacao': f"r={pearson_r:.3f}{significancia}"
    }

print("Função de correlação por cluster definida.")

In [ ]:
# Calcular correlações para todos os clusters e variáveis IDH
clusters = sorted(df['cluster'].dropna().unique())

resultados = []

for cluster_id in clusters:
    for idh_var in IDH_VARS:
        resultado = calcular_correlacao_por_cluster(df, cluster_id, CORRUPTION_VAR, idh_var)
        resultado['idh_var'] = idh_var
        resultados.append(resultado)

df_resultados = pd.DataFrame(resultados)

print("\nTabela de Correlações por Cluster")
print("="*80)

# Pivot para visualização
pivot_table = df_resultados.pivot_table(
    index=['cluster', 'n_municipios'], 
    columns='idh_var', 
    values='interpretacao',
    aggfunc='first'
)

print(pivot_table)

In [ ]:
# Visualização das correlações por cluster
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, idh_var in enumerate(IDH_VARS):
    ax = axes[idx]
    
    subset = df_resultados[df_resultados['idh_var'] == idh_var].copy()
    
    # Criar barras coloridas baseado na significância
    colors = []
    for _, row in subset.iterrows():
        if row['pearson_p'] < 0.001:
            colors.append('darkred')
        elif row['pearson_p'] < 0.01:
            colors.append('red')
        elif row['pearson_p'] < 0.05:
            colors.append('orange')
        else:
            colors.append('lightgray')
    
    bars = ax.bar(subset['cluster'], subset['pearson_r'], color=colors, alpha=0.8, edgecolor='black')
    
    # Adicionar linha de referência em r=0
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    
    # Labels
    ax.set_xlabel('Cluster')
    ax.set_ylabel("Correlação de Pearson (r)")
    ax.set_title(f"{idh_var}\nvs Sanções/Milhão BRL")
    ax.set_ylim(-0.5, 0.5)
    
    # Adicionar valores nas barras
    for bar, r, p in zip(bars, subset['pearson_r'], subset['pearson_p']):
        height = bar.get_height()
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ax.text(
            bar.get_x() + bar.get_width()/2.,
            height + (0.02 if height >= 0 else -0.05),
            f"{r:.3f}{sig}",
            ha='center', va='bottom' if height >= 0 else 'top',
            fontsize=9, fontweight='bold'
        )

# Legenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='darkred', label='p < 0.001 ***'),
    Patch(facecolor='red', label='p < 0.01 **'),
    Patch(facecolor='orange', label='p < 0.05 *'),
    Patch(facecolor='lightgray', label='não significativo')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 0.98))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.suptitle("Correlação: Sanções/Transferência vs Indicadores de Desenvolvimento\npor Cluster", fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 4. Índice de Vulnerabilidade Municipal

Criamos um índice combinado para visualização no mapa:
- **Alto índice (vermelho):** Muitas sanções por recurso transferido + Baixo IDH = Alta vulnerabilidade/corrupção
- **Baixo índice (azul):** Poucas sanções por recurso + Alto IDH = Baixa vulnerabilidade/boa gestão

In [ ]:
# Criar índice de vulnerabilidade
def calcular_indice_vulnerabilidade(row):
    """
    Índice combinado: (sanções normalizadas) - (IDH normalizado)
    Quanto maior, mais vulnerável (alta corrupção, baixo desenvolvimento)
    """
    
    # Normalizar sanções (0-1, onde 1 = mais sanções)
    if pd.isna(row[CORRUPTION_VAR]) or row[CORRUPTION_VAR] < 0:
        return None
    
    # Usar log para reduzir skewness
    sanctions_log = np.log1p(row[CORRUPTION_VAR])
    
    # Componente IDH (média de renda e alfabetização normalizadas)
    income_norm = row['avg_income_real_2022_2022_brl'] if pd.notna(row['avg_income_real_2022_2022_brl']) else 0
    literacy_norm = row['literacy_rate_2022'] if pd.notna(row['literacy_rate_2022']) else 0
    
    # IDH score (0-1 aproximado)
    idh_score = (income_norm / 5000 + literacy_norm / 100) / 2  # Normalização aproximada
    idh_score = min(max(idh_score, 0), 1)  # Clip 0-1
    
    # Vulnerabilidade: corrupção - desenvolvimento
    # Quanto maior, pior (mais corrupção relativa, menos desenvolvimento)
    vulnerability = sanctions_log - (idh_score * 5)  # Dar peso ao IDH
    
    return vulnerability

# Aplicar a todos os municípios com dados
df['vulnerability_index'] = df.apply(calcular_indice_vulnerabilidade, axis=1)

# Estatísticas do índice
print("Estatísticas do Índice de Vulnerabilidade:")
print(df['vulnerability_index'].describe())

# Distribuição por cluster
print("\nDistribuição do Índice por Cluster:")
print(df.groupby('cluster')['vulnerability_index'].describe().round(3))

In [ ]:
# Visualização da distribuição do índice por cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot por cluster
ax1 = axes[0]
cluster_data = [df[df['cluster'] == c]['vulnerability_index'].dropna() for c in sorted(df['cluster'].dropna().unique())]
bp = ax1.boxplot(cluster_data, labels=[f'Cluster {int(c)}' for c in sorted(df['cluster'].dropna().unique())], patch_artist=True)

# Colorir caixas
colors = ['lightblue', 'lightgreen', 'lightyellow', 'lightcoral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax1.set_xlabel('Cluster')
ax1.set_ylabel('Índice de Vulnerabilidade')
ax1.set_title('Distribuição do Índice por Cluster')
ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Referência neutra')
ax1.legend()

# Histograma geral
ax2 = axes[1]
ax2.hist(df['vulnerability_index'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.7)
ax2.set_xlabel('Índice de Vulnerabilidade')
ax2.set_ylabel('Frequência')
ax2.set_title('Distribuição Geral do Índice')
ax2.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Seleção de Amostra Representativa

Selecionar municípios extremos (melhores e piores) dentro de cada cluster para apresentação.

In [ ]:
# Selecionar top N por cluster (melhores e piores no índice de vulnerabilidade)
TOP_N = 5

amostra_representativa = []

for cluster_id in sorted(df['cluster'].dropna().unique()):
    cluster_df = df[df['cluster'] == cluster_id].copy()
    
    # Melhores (menor vulnerabilidade = azul)
    melhores = cluster_df.nsmallest(TOP_N, 'vulnerability_index')
    melhores['categoria'] = 'Melhor Gestão'
    melhores['ranking_no_cluster'] = range(1, len(melhores) + 1)
    
    # Piores (maior vulnerabilidade = vermelho)
    piores = cluster_df.nlargest(TOP_N, 'vulnerability_index')
    piores['categoria'] = 'Maior Vulnerabilidade'
    piores['ranking_no_cluster'] = range(1, len(piores) + 1)
    
    amostra_representativa.extend([melhores, piores])

df_amostra = pd.concat(amostra_representativa, ignore_index=True)

print(f"Amostra representativa selecionada: {len(df_amostra)} municípios")
print(f"  - {len(df_amostra[df_amostra['categoria'] == 'Melhor Gestão'])} melhores")
print(f"  - {len(df_amostra[df_amostra['categoria'] == 'Maior Vulnerabilidade'])} piores")

# Colunas para exibição
display_cols = [
    'municipality_code', 'municipality_name', 'state_name', 'cluster',
    'categoria', 'ranking_no_cluster',
    'sanctions_per_million_brl_transfers', 'avg_income_real_2022_2022_brl', 
    'literacy_rate_2022', 'vulnerability_index'
]

print("\nPrimeiros casos da amostra:")
print(df_amostra[display_cols].head(10).to_string(index=False))

In [ ]:
# Visualização da amostra em coordenadas PCA
fig = px.scatter(
    df,
    x='PC1', y='PC2',
    color='vulnerability_index',
    color_continuous_scale='RdYlBu_r',  # Vermelho = alto (ruim), Azul = baixo (bom)
    hover_data=['municipality_name', 'state_name', 'sanctions_per_million_brl_transfers', 'avg_income_real_2022_2022_brl'],
    title='Municípios no Espaço PCA (colorido por Índice de Vulnerabilidade)',
    labels={'PC1': 'Componente Principal 1', 'PC2': 'Componente Principal 2', 'vulnerability_index': 'Vulnerabilidade'}
)

# Adicionar destaque para amostra representativa
amostra_coords = df_amostra[['PC1', 'PC2', 'municipality_name', 'categoria']].copy()

fig.add_trace(
    go.Scatter(
        x=amostra_coords[amostra_coords['categoria'] == 'Melhor Gestão']['PC1'],
        y=amostra_coords[amostra_coords['categoria'] == 'Melhor Gestão']['PC2'],
        mode='markers',
        marker=dict(size=12, color='blue', symbol='star', line=dict(width=2, color='black')),
        name='Melhor Gestão (Amostra)',
        text=amostra_coords[amostra_coords['categoria'] == 'Melhor Gestão']['municipality_name'],
        hovertemplate='%{text}<extra></extra>'
    )
)

fig.add_trace(
    go.Scatter(
        x=amostra_coords[amostra_coords['categoria'] == 'Maior Vulnerabilidade']['PC1'],
        y=amostra_coords[amostra_coords['categoria'] == 'Maior Vulnerabilidade']['PC2'],
        mode='markers',
        marker=dict(size=12, color='red', symbol='x', line=dict(width=2, color='black')),
        name='Maior Vulnerabilidade (Amostra)',
        text=amostra_coords[amostra_coords['categoria'] == 'Maior Vulnerabilidade']['municipality_name'],
        hovertemplate='%{text}<extra></extra>'
    )
)

fig.update_layout(height=700)
fig.show()

## 6. Geração de GeoJSON para QGIS

Criar arquivo GeoJSON com o índice de vulnerabilidade para visualização no QGIS.

In [ ]:
# Preparar dados para GeoJSON
# Juntar com dados geográficos (centroides dos municípios)

from pathlib import Path
import zipfile
import tempfile

try:
    import shapefile  # pyshp
except ImportError as exc:
    raise ImportError("pyshp é necessário. Instale com: pip install pyshp>=2.3.1") from exc

# Carregar shapefile dos municípios
map_assets_dir = Path("..") / "docs" / "thesis_presentation_assets" / "qgis"
municipality_zip_path = map_assets_dir / "BR_Municipios_2022.zip"

print(f"Carregando shapefile: {municipality_zip_path}")

with tempfile.TemporaryDirectory(prefix="ibge_muni_shape_") as _tmp_dir:
    with zipfile.ZipFile(municipality_zip_path) as _zip_file:
        _zip_file.extractall(_tmp_dir)

    _shp_path = next(Path(_tmp_dir).glob("*.shp"))
    _reader = shapefile.Reader(str(_shp_path))
    _fields = [field[0] for field in _reader.fields[1:]]
    _code_idx = _fields.index("CD_MUN")

    # Extrair centroides
    centroides = []
    for _shape_record in _reader.iterShapeRecords():
        municipality_code = str(_shape_record.record[_code_idx]).zfill(7)
        xmin, ymin, xmax, ymax = _shape_record.shape.bbox
        centroides.append({
            "municipality_code": municipality_code,
            "lon": (xmin + xmax) / 2.0,
            "lat": (ymin + ymax) / 2.0,
        })
    _reader.close()  # Close before temp cleanup (Windows fix)

df_centroides = pd.DataFrame(centroides)
print(f"Centroides carregados: {len(df_centroides):,}")

# Juntar com dados de análise
df_geo = df.merge(df_centroides, on='municipality_code', how='inner')
print(f"Municípios com dados de análise + coordenadas: {len(df_geo):,}")

In [ ]:
# Criar categorias de vulnerabilidade para o mapa
def categorizar_vulnerabilidade(v):
    if pd.isna(v):
        return 'Sem dados'
    elif v < -3:
        return 'Muito Baixa (Azul)'
    elif v < -1:
        return 'Baixa (Azul Claro)'
    elif v < 1:
        return 'Neutra (Amarelo)'
    elif v < 3:
        return 'Alta (Laranja)'
    else:
        return 'Muito Alta (Vermelho)'

df_geo['vulnerability_category'] = df_geo['vulnerability_index'].apply(categorizar_vulnerabilidade)

print("Distribuição das categorias:")
print(df_geo['vulnerability_category'].value_counts())

In [ ]:
# Criar GeoJSON para QGIS
import json

features = []

for _, row in df_geo.iterrows():
    if pd.isna(row['lon']) or pd.isna(row['lat']):
        continue
    
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [float(row['lon']), float(row['lat'])]
        },
        "properties": {
            "municipality_code": str(row['municipality_code']),
            "municipality_name": str(row['municipality_name']),
            "state_code": str(row['state_code']) if pd.notna(row['state_code']) else None,
            "state_name": str(row['state_name']) if pd.notna(row['state_name']) else None,
            "cluster": int(row['cluster']) if pd.notna(row['cluster']) else None,
            "vulnerability_index": float(row['vulnerability_index']) if pd.notna(row['vulnerability_index']) else None,
            "vulnerability_category": str(row['vulnerability_category']),
            "sanctions_per_million": float(row['sanctions_per_million_brl_transfers']) if pd.notna(row['sanctions_per_million_brl_transfers']) else 0,
            "avg_income_2022": float(row['avg_income_real_2022_2022_brl']) if pd.notna(row['avg_income_real_2022_2022_brl']) else None,
            "literacy_rate_2022": float(row['literacy_rate_2022']) if pd.notna(row['literacy_rate_2022']) else None,
            "total_transfers": float(row['total_transfers']) if pd.notna(row['total_transfers']) else 0,
            "n_sanctions": int(row['n_sanctions']) if pd.notna(row['n_sanctions']) else 0,
        }
    }
    features.append(feature)

geojson = {
    "type": "FeatureCollection",
    "name": "Municipalities_Vulnerability_Index",
    "crs": {
        "type": "name",
        "properties": {
            "name": "urn:ogc:def:crs:OGC:1.3:CRS84"
        }
    },
    "features": features
}

# Salvar
output_path = map_assets_dir / "brazil_municipalities_vulnerability_index.geojson"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)

print(f"GeoJSON salvo: {output_path}")
print(f"Total de features: {len(features)}")

# Estatísticas por estado
print("\nTop 10 estados com maior média de vulnerabilidade:")
state_vuln = df_geo.groupby('state_name')['vulnerability_index'].mean().sort_values(ascending=False).head(10)
print(state_vuln.round(3))

## 7. Dashboard Consolidado

In [ ]:
# Resumo executivo
print("="*80)
print("RESUMO EXECUTIVO - ANÁLISE DE CORRUPÇÃO/MAU USO vs IDH")
print("="*80)

print(f"\n📊 Dados Analisados:")
print(f"   • Total de municípios: {len(df):,}")
print(f"   • Com dados de sanções: {df['has_sanctions_data'].sum():,}")
print(f"   • Clusters definidos: {len(clusters)}")

print(f"\n📈 Correlações Significativas (p < 0.05):")
sig_corrs = df_resultados[df_resultados['pearson_p'] < 0.05]
for _, row in sig_corrs.iterrows():
    print(f"   • Cluster {int(row['cluster'])} vs {row['idh_var']}: r={row['pearson_r']:.3f} {row['significancia']}")

if len(sig_corrs) == 0:
    print("   • Nenhuma correlação significativa detectada nos clusters")
    print("   • Isso sugere que o efeito é heterogêneo ou de baixa magnitude")

print(f"\n🎯 Amostra Representativa:")
print(f"   • {TOP_N} melhores municípios por cluster = {len(df_amostra[df_amostra['categoria'] == 'Melhor Gestão'])} total")
print(f"   • {TOP_N} piores municípios por cluster = {len(df_amostra[df_amostra['categoria'] == 'Maior Vulnerabilidade'])} total")

print(f"\n🗺️  Assets Gerados:")
print(f"   • GeoJSON municipal: docs/thesis_presentation_assets/qgis/brazil_municipalities_vulnerability_index.geojson")
print(f"   • {len(features)} municípios com coordenadas e índice de vulnerabilidade")

print("\n" + "="*80)

In [ ]:
# Exportar tabelas para dashboard

# 1. Tabela de correlações
correlations_export = df_resultados[
    ['cluster', 'n_municipios', 'idh_var', 'pearson_r', 'pearson_p', 'significancia', 'interpretacao']
].copy()

# 2. Tabela da amostra representativa
amostra_export = df_amostra[
    ['municipality_code', 'municipality_name', 'state_code', 'state_name', 
     'cluster', 'categoria', 'ranking_no_cluster', 'vulnerability_index',
     'sanctions_per_million_brl_transfers', 'avg_income_real_2022_2022_brl', 
     'literacy_rate_2022', 'total_transfers', 'n_sanctions']
].copy()

# 3. Resumo por cluster
resumo_cluster = df.groupby('cluster').agg({
    'municipality_code': 'count',
    'vulnerability_index': ['mean', 'std', 'min', 'max'],
    'sanctions_per_million_brl_transfers': ['mean', 'median'],
    'avg_income_real_2022_2022_brl': 'mean',
    'literacy_rate_2022': 'mean'
}).round(3)

# Salvar CSVs
output_dir = Path("..") / "docs" / "thesis_presentation_assets"

correlations_export.to_csv(output_dir / "correlation_by_cluster.csv", index=False)
amostra_export.to_csv(output_dir / "amostra_representativa_cidades.csv", index=False)
resumo_cluster.to_csv(output_dir / "resumo_por_cluster.csv")

print("Tabelas exportadas:")
print(f"  1. {output_dir / 'correlation_by_cluster.csv'}")
print(f"  2. {output_dir / 'amostra_representativa_cidades.csv'}")
print(f"  3. {output_dir / 'resumo_por_cluster.csv'}")

---

## Fim da Análise

### Principais Conclusões

1. **Correlação por Cluster:** A relação entre sanções/transferência e indicadores de desenvolvimento varia significativamente entre clusters, refletindo diferentes realidades regionais.

2. **Índice de Vulnerabilidade:** O índice combinado permite identificar municípios com alta ineficiência relativa (sanções por recurso) mesmo dentro de contextos socioeconômicos similares.

3. **Amostra Representativa:** A seleção de extremos (melhores/piores) por cluster fornece casos concretos para análise qualitativa e apresentação.

4. **Mapa para QGIS:** O GeoJSON gerado permite visualização espacial das vulnerabilidades, identificando hotspots geográficos de maior risco.

### Próximos Passos

- **Análise Qualitativa:** Investigar casos extremos da amostra representativa para entender fatores contextuais
- **Mapa no QGIS:** Abrir o GeoJSON no QGIS e aplicar simbologia graduada pelo índice de vulnerabilidade
- **Dashboard Interativo:** Usar as tabelas exportadas para criar visualizações interativas (Power BI, Tableau, etc.)
- **Validação:** Consultar literatura sobre métricas similares de eficiência governamental